# Score notes as a bopp extension

Brian McFee's PR #8 (bmcfee/bopp) adds `ext` payloads, whose items are validated in a second pass by a type that an installed package registers under the `bopp_extension` entry-point group. The `bopp-score-ext` plugin in `experiments/plugin/` registers `io.github.johentsch.score_note:v1` as a `ScoreNote` struct that mirrors the columnar `score_note` payload. This notebook builds a `score_note` annotation from a Distant Listening Corpus note table, re-encodes it as an `ext` annotation, and shows the ids, file sizes, the no-plugin fallback, validation errors and timings.

**Setup**
- Needs the plugin installed (`pip install -e experiments/plugin`), otherwise the schema is not in the registry.
- Reads 822 notes from a DLC fixture into a columnar `score_note` annotation, the baseline to compare against.

In [1]:
import tempfile
import timeit
import warnings
from pathlib import Path

import msgspec
import pandas as pd
from bopp_score_ext import EXT_SCHEMA, from_ext, to_ext

import bopp
import bopp.core
import bopp.extensions
import bopp.util
from bopp.io import load_bopp_msgpack, save_bopp_json, save_bopp_msgpack
from experiments.dlc_notes import notes_to_score_note, read_notes_tsv

# bopp is imported from the repository checkout, so the fixture is located from the package,
# not from the kernel working directory (the notebook may be run from a copy elsewhere)
REPO = Path(bopp.__file__).resolve().parents[1]
FIXTURE = REPO / "tests" / "experiments" / "data" / "dlc_notes_chopin_mazurkas_BI163op67-4.tsv"
WORK = Path(tempfile.mkdtemp())

ann = notes_to_score_note(read_notes_tsv(FIXTURE))
bopp.util.to_dataframe(ann).head(3)

,payload:score_note:midi,payload:score_note:tpc,payload:score_note:name,payload:score_note:octave,payload:score_note:staff,payload:score_note:voice,payload:score_note:mc,payload:score_note:mn,payload:score_note:tied,payload:score_note:gracenote,payload:score_note:nominal_duration,payload:score_note:scalar,payload:score_note:chord_id,payload:score_note:volta,extent:quarters_interval.fraction:quarter,extent:quarters_interval.fraction:duration
0,76,4,E5,5,1,1,1,0,1.0,NaN,"[1, 4]","[1, 1]",0,NaN,"[0, 1]","[1, 1]"
1,45,3,A2,2,2,1,2,1,NaN,NaN,"[1, 4]","[1, 1]",5,NaN,"[1, 1]","[1, 1]"
2,76,4,E5,5,1,1,2,1,-1.0,NaN,"[1, 4]","[3, 2]",1,NaN,"[1, 1]","[3, 2]"


**Columnar → `ext`**
- `to_ext` turns the columns into one `ScoreNote` row per note under the plugin's `ext_schema`.
- The two forms are different payloads, so they get different ids.

In [2]:
ext = to_ext(ann)
{
    "ext_schema": ext.payload.ext_schema,
    "first row": ext.payload.value[0],
    "columnar id": ann.id,
    "ext id": ext.id,
}

{'ext_schema': 'io.github.johentsch.score_note:v1',
 'first row': ScoreNote(midi=76, tpc=4, name='E5', octave=5, staff=1, voice=1, mc=1, mn=0, measure=UNSET, tied=1, gracenote=UNSET, nominal_duration=FractionPair(numerator=1, denominator=4), scalar=FractionPair(numerator=1, denominator=1), chord_id=0, tremolo=UNSET, volta=UNSET, tuning=UNSET, anchor=UNSET),
 'columnar id': '9e09e71c-b72a-5ea7-af07-430412d1d877',
 'ext id': '0708a693-7d74-5267-abe0-7c04fa7ea9d9'}

**File sizes**
- Saves both forms as JSON and MsgPack. The row form repeats the field names in every row, so its files are larger.

In [3]:
save_bopp_json(ann, WORK / "columnar.json")
save_bopp_msgpack(ann, WORK / "columnar.msgpack")
save_bopp_json(ext, WORK / "ext.json")
save_bopp_msgpack(ext, WORK / "ext.msgpack")
pd.DataFrame(
    {fmt: {form: (WORK / f"{form}.{fmt}").stat().st_size for form in ("columnar", "ext")} for fmt in ("json", "msgpack")}
).rename_axis(index="payload", columns="bytes")

bytes,json,msgpack
payload,,
columnar,54919,23490
ext,122046,82927


**Ids with and without resolution**
- Loads the `ext` file with rows resolved to `ScoreNote` and left as dicts.
- Both give the same, valid id, so the id doesn't depend on whether the plugin is present.

In [4]:
resolved = load_bopp_msgpack(WORK / "ext.msgpack")
unresolved = load_bopp_msgpack(WORK / "ext.msgpack", resolve_ext=False)
{
    "first row, resolved": type(resolved.payload.value[0]).__name__,
    "first row, unresolved": type(unresolved.payload.value[0]).__name__,
    "same id": ext.id == resolved.id == unresolved.id,
    "ids valid": bopp.core.validate_annotation_id(resolved) and bopp.core.validate_annotation_id(unresolved),
}

{'first row, resolved': 'ScoreNote',
 'first row, unresolved': 'dict',
 'same id': True,
 'ids valid': True}

**Plugin missing**
- Empties the registry to simulate a reader without the plugin.
- Loading warns and keeps the rows as dicts. The file still loads and its id is unchanged.

In [5]:
bopp.extensions.REGISTRY.clear()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    without_plugin = load_bopp_msgpack(WORK / "ext.msgpack")
bopp.extensions.reset_extensions()
{
    "warning": next(str(w.message) for w in caught if "No extension registered" in str(w.message)),
    "row type": type(without_plugin.payload.value[0]).__name__,
    "id unchanged": without_plugin.id == ext.id,
    "registered again": EXT_SCHEMA in bopp.extensions.get_extensions(),
}

{'warning': "No extension registered for schema 'io.github.johentsch.score_note:v1'. Skipping validation.",
 'row type': 'dict',
 'id unchanged': True,
 'registered again': True}

**Validation**
- With the plugin registered, bad rows are rejected when the annotation is created: an out-of-range `midi` value and an unknown field.

In [6]:
def creation_error(row):
    """The validation error raised when creating an ext annotation from ``row``, if any."""
    try:
        bopp.create(media_id="demo:bad", payload_kind="ext", ext_schema=EXT_SCHEMA, value=[row])
    except msgspec.ValidationError as err:
        return str(err)
    return None


{
    "midi out of range": creation_error({"midi": 128, "tpc": 0}),
    "unknown field": creation_error({"midi": 60, "tpc": 0, "unknown": 1}),
}

{'midi out of range': 'Expected `int` <= 127 - at `$[0].midi`',
 'unknown field': 'Object contains unknown field `unknown` - at `$[0]`'}

**Round trip**
- `from_ext` turns the rows back into columns and reproduces the original columnar id.

In [7]:
from_ext(to_ext(ann)).id == ann.id

True

**Timing**
- MsgPack decode time for the columnar form, the `ext` form without row validation, and the `ext` form with it.

In [8]:
def best_ms(stmt):
    """Best of 5 repeats of 20 calls, per call, in milliseconds."""
    return min(timeit.repeat(stmt, number=20, repeat=5)) / 20 * 1000


pd.Series(
    {
        "columnar": best_ms(lambda: load_bopp_msgpack(WORK / "columnar.msgpack")),
        "ext, first pass only": best_ms(lambda: load_bopp_msgpack(WORK / "ext.msgpack", resolve_ext=False)),
        "ext, both passes": best_ms(lambda: load_bopp_msgpack(WORK / "ext.msgpack")),
    },
    name="msgpack decode (ms)",
).round(3)

columnar                1.713
ext, first pass only    2.710
ext, both passes        2.530
Name: msgpack decode (ms), dtype: float64

The `ext` form has the same id with the plugin resolved, with rows left as dicts, and after a JSON or MsgPack round trip, because the rows serialize to the same bytes either way. Row validation is the second pass: it runs only when the plugin is registered, so a file stays readable without it and its id still checks out. The second pass (row validation) adds no measurable time here: the `ext` timings with and without it are within run-to-run noise. The `ext` file is larger and slower to decode than the columnar one, so the extra cost comes from the row-oriented first pass, not from validation. Two caveats from the plugin README remain: CSV round trips of `ext` payloads are not id-stable upstream yet, and a column that is entirely `None` does not survive `to_ext`.

# Score control events as a bopp extension

The DLC chords facet lists seven kinds of score control events (chord, dynamic, spanner, figured_bass, staff_text, system_text, tempo), one homogeneous columnar `score_control_event.<kind>` payload per kind per piece. The `bopp-score-ext` plugin offers two `ext` encodings: one row-typed `ext` annotation per event kind, registered under `io.github.johentsch.score_control_event.<kind>:v1`, and one mixed `ext` annotation holding all events of the piece in score order, tagged by an `event` key, under the unregistered `io.github.johentsch.score_control_event:v1` schema. This part repeats the ids, sizes, validation and timing comparisons from the notes part for both encodings, and shows why the mixed schema cannot be registered as an entry point.

**Setup**
- Reads the chord-facet events of the same mazurka into one columnar annotation per event kind.
- `chords_to_control_events` groups the rows by kind and keeps TSV order within each group.

In [9]:
from bopp_score_ext import (
    EVENT_EXT_SCHEMAS,
    MIXED_EXT_SCHEMA,
    ScoreControlEvent,
    decode_mixed,
)

from experiments.dlc_chords import (
    chords_to_control_events,
    chords_to_mixed_ext,
    read_chords_tsv,
)

CHORDS_FIXTURE = REPO / "tests" / "experiments" / "data" / "dlc_chords_chopin_mazurkas_BI163op67-4.tsv"

chords = read_chords_tsv(CHORDS_FIXTURE)
events = chords_to_control_events(chords)
KINDS = tuple(events)

pd.DataFrame(
    {
        kind: {
            "payload class": type(events[kind].payload).__name__,
            "rows": len(events[kind].payload.staff),
        }
        for kind in KINDS
    }
).T

,payload class,rows
chord,ScoreChordPayload,563
dynamic,ScoreDynamicPayload,5
spanner,ScoreSpannerPayload,132
staff_text,ScoreStaffTextPayload,14
tempo,ScoreTempoPayload,2


**Columnar → `ext` per type**
- `to_ext` turns each kind's columns into one row per event under its own `ext_schema`.
- The resolved row type is the plugin's dedicated struct (`ScoreChord`, ...), not the columnar payload type.

In [10]:
ext_by_kind = {kind: to_ext(events[kind]) for kind in KINDS}
{
    "ext_schema": {kind: ext_by_kind[kind].payload.ext_schema for kind in KINDS},
    "row type": {kind: type(ext_by_kind[kind].payload.value[0]).__name__ for kind in KINDS},
    "chord[0]": ext_by_kind["chord"].payload.value[0],
    "tempo[0]": ext_by_kind["tempo"].payload.value[0],
}

{'ext_schema': {'chord': 'io.github.johentsch.score_control_event.chord:v1',
  'dynamic': 'io.github.johentsch.score_control_event.dynamic:v1',
  'spanner': 'io.github.johentsch.score_control_event.spanner:v1',
  'staff_text': 'io.github.johentsch.score_control_event.staff_text:v1',
  'tempo': 'io.github.johentsch.score_control_event.tempo:v1'},
 'row type': {'chord': 'ScoreChord',
  'dynamic': 'ScoreDynamic',
  'spanner': 'ScoreSpanner',
  'staff_text': 'ScoreStaffText',
  'tempo': 'ScoreTempo'},
 'chord[0]': ScoreChord(staff=1, voice=1, mc=1, mn=0, chord_id=0, volta=UNSET, slur=UNSET, crescendo_hairpin=UNSET, decrescendo_hairpin=UNSET, crescendo_line=UNSET, diminuendo_line=UNSET, pedal=UNSET, nominal_duration=FractionPair(numerator=1, denominator=4), scalar=FractionPair(numerator=1, denominator=1), gracenote=UNSET, articulation='articAccentAbove', tremolo=UNSET, lyrics_1=UNSET, lyrics_2=UNSET, lyrics_3=UNSET),
 'tempo[0]': ScoreTempo(staff=1, voice=1, mc=1, mn=0, qpm=114.0, tempo_vis

**Mixed `ext`**
- `chords_to_mixed_ext` builds one annotation with every event of the piece in score order, tagged by `event`.
- `decode_mixed` strictly decodes the tagged union without registering or mutating anything.

In [11]:
mixed = chords_to_mixed_ext(chords)
decoded = decode_mixed(mixed)
{
    "ext_schema": mixed.payload.ext_schema,
    "rows": len(mixed.payload.value),
    "first 4 event tags": [row["event"] for row in mixed.payload.value[:4]],
    "first 4 decoded types": [type(e).__name__ for e in decoded[:4]],
    "re-encodes to same bytes": msgspec.json.encode(decoded) == msgspec.json.encode(mixed.payload.value),
}

{'ext_schema': 'io.github.johentsch.score_control_event:v1',
 'rows': 716,
 'first 4 event tags': ['Tempo', 'Tempo', 'Chord', 'Dynamic'],
 'first 4 decoded types': ['TempoEvent',
  'TempoEvent',
  'ChordEvent',
  'DynamicEvent'],
 're-encodes to same bytes': True}

**Why mixed is not registered**
- An entry point, or `REGISTRY.register_entry` with a module path, resolves the same way and fails on lookup because the registry requires a type, not a union.
- Assigning the union directly to the registry bypasses that check; msgspec then resolves the tagged union fine. The registration is removed again afterwards either way.

In [12]:
try:
    bopp.extensions.REGISTRY.register_entry(MIXED_EXT_SCHEMA, "bopp_score_ext.events:ScoreControlEvent")
    bopp.extensions.REGISTRY[MIXED_EXT_SCHEMA]
except TypeError as err:
    entry_point_error = str(err)
del bopp.extensions.REGISTRY[MIXED_EXT_SCHEMA]

mixed_msgpack = WORK / "mixed.msgpack"
save_bopp_msgpack(mixed, mixed_msgpack)
bopp.extensions.REGISTRY[MIXED_EXT_SCHEMA] = ScoreControlEvent
bypassed = load_bopp_msgpack(mixed_msgpack)
bypassed_types = [type(row).__name__ for row in bypassed.payload.value[:4]]
del bopp.extensions.REGISTRY[MIXED_EXT_SCHEMA]
{
    "entry point error": entry_point_error,
    "bypass first 4 row types": bypassed_types,
    "bypass id == unresolved id": bypassed.id == mixed.id,
    "mixed schema removed again": MIXED_EXT_SCHEMA not in bopp.extensions.REGISTRY,
}

{'entry point error': "Resolved extension for 'io.github.johentsch.score_control_event:v1' must be a type, got UnionType",
 'bypass first 4 row types': ['TempoEvent',
  'TempoEvent',
  'ChordEvent',
  'DynamicEvent'],
 'bypass id == unresolved id': True,
 'mixed schema removed again': True}

**File sizes**
- Saves three forms: the per-kind columnar annotations, the per-kind `ext` annotations, and the one mixed `ext` annotation.
- Sums bytes across the per-kind files for the first two forms; the mixed form is already one file.

In [13]:
for kind in KINDS:
    save_bopp_json(events[kind], WORK / f"{kind}.json")
    save_bopp_msgpack(events[kind], WORK / f"{kind}.msgpack")
    save_bopp_json(ext_by_kind[kind], WORK / f"ext_{kind}.json")
    save_bopp_msgpack(ext_by_kind[kind], WORK / f"ext_{kind}.msgpack")
save_bopp_json(mixed, WORK / "mixed.json")

pd.DataFrame(
    {
        fmt: {
            "columnar": sum((WORK / f"{kind}.{fmt}").stat().st_size for kind in KINDS),
            "ext per type": sum((WORK / f"ext_{kind}.{fmt}").stat().st_size for kind in KINDS),
            "ext mixed": (WORK / f"mixed.{fmt}").stat().st_size,
        }
        for fmt in ("json", "msgpack")
    }
).rename_axis(index="form", columns="bytes")

bytes,json,msgpack
form,,
columnar,57809,21434
ext per type,78515,53728
ext mixed,89098,60459


**Ids with and without resolution**
- Loads every per-kind `ext` file resolved and with `resolve_ext=False`; both give the same, valid id.
- Loading the mixed file with default resolution warns, since its schema is not registered; rows stay dicts and the id is unchanged.

In [14]:
ids_match = {}
for kind in KINDS:
    resolved = load_bopp_msgpack(WORK / f"ext_{kind}.msgpack")
    unresolved = load_bopp_msgpack(WORK / f"ext_{kind}.msgpack", resolve_ext=False)
    valid = bopp.core.validate_annotation_id(resolved) and bopp.core.validate_annotation_id(unresolved)
    ids_match[kind] = ext_by_kind[kind].id == resolved.id == unresolved.id and valid

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    mixed_loaded = load_bopp_msgpack(WORK / "mixed.msgpack")
{
    "ids valid for every kind": all(ids_match.values()),
    "mixed warning": next(str(w.message) for w in caught if "No extension registered" in str(w.message)),
    "mixed rows stay dicts": all(type(row) is dict for row in mixed_loaded.payload.value),
    "mixed id unchanged": mixed_loaded.id == mixed.id,
}

{'ids valid for every kind': True,
 'mixed warning': "No extension registered for schema 'io.github.johentsch.score_control_event:v1'. Skipping validation.",
 'mixed rows stay dicts': True,
 'mixed id unchanged': True}

**Plugin missing**
- Empties the registry to simulate a reader without the plugin, then loads all the per-kind `ext` files.
- Each load warns and keeps its id; `reset_extensions` brings the per-kind schemas back.

In [15]:
bopp.extensions.REGISTRY.clear()
missing = {}
for kind in KINDS:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        loaded = load_bopp_msgpack(WORK / f"ext_{kind}.msgpack")
    missing[kind] = (
        any("No extension registered" in str(w.message) for w in caught)
        and loaded.id == ext_by_kind[kind].id
    )
bopp.extensions.reset_extensions()
{
    "warned and id kept, every kind": all(missing.values()),
    "schemas back": all(s in bopp.extensions.get_extensions() for s in EVENT_EXT_SCHEMAS.values()),
}

{'warned and id kept, every kind': True, 'schemas back': True}

**Validation**
- Rejected rows through `bopp.create(payload_kind="ext", ...)`: a tempo with `qpm` 0.0, a chord with an unknown field, a chord with an explicit `slur: None`.
- `decode_mixed` rejects a row with an unknown `event` tag the same way.

In [16]:
def event_creation_error(ext_schema, row):
    """The validation error raised when creating an ext annotation from ``row``, if any."""
    try:
        bopp.create(media_id="demo:bad-event", payload_kind="ext", ext_schema=ext_schema, value=[row])
    except msgspec.ValidationError as err:
        return str(err)
    return None

def mixed_decode_error(row):
    """The validation error raised when decode_mixed sees ``row``, if any."""
    one_row_extent = msgspec.structs.replace(
        mixed.extent, quarter=mixed.extent.quarter[:1], duration=mixed.extent.duration[:1]
    )
    bad = msgspec.structs.replace(
        mixed, extent=one_row_extent, payload=msgspec.structs.replace(mixed.payload, value=[row])
    )
    try:
        decode_mixed(bad)
    except msgspec.ValidationError as err:
        return str(err)
    return None

chord_row = msgspec.to_builtins(ext_by_kind["chord"].payload.value[0])
tempo_row = msgspec.to_builtins(ext_by_kind["tempo"].payload.value[0])
bad_tempo = {**tempo_row, "qpm": 0.0}
bad_chord_unknown = {**chord_row, "unknown": 1}
bad_chord_null = {**chord_row, "slur": None}
bad_mixed_row = {**mixed.payload.value[0], "event": "Unknown"}

{
    "tempo qpm=0.0": event_creation_error(EVENT_EXT_SCHEMAS["tempo"], bad_tempo),
    "chord unknown field": event_creation_error(EVENT_EXT_SCHEMAS["chord"], bad_chord_unknown),
    "chord slur=None": event_creation_error(EVENT_EXT_SCHEMAS["chord"], bad_chord_null),
    "decode_mixed unknown event": mixed_decode_error(bad_mixed_row),
}

{'tempo qpm=0.0': 'Expected `float` > 0.0 - at `$[0].qpm`',
 'chord unknown field': 'Object contains unknown field `unknown` - at `$[0]`',
 'chord slur=None': 'Expected `str`, got `null` - at `$[0].slur`',
 'decode_mixed unknown event': "Invalid value 'Unknown' - at `$[0].event`"}

**Round trip**
- `from_ext(to_ext(a)).id == a.id` for every event kind.

In [17]:
{kind: from_ext(to_ext(events[kind])).id == events[kind].id for kind in KINDS}

{'chord': True,
 'dynamic': True,
 'spanner': True,
 'staff_text': True,
 'tempo': True}

**Timing**
- MsgPack decode time summed over the per-kind files: the columnar form, the `ext` form without row validation, and the `ext` form with it.
- The mixed form's first pass alone, and first pass followed by `decode_mixed`.

In [18]:
columnar_paths = [WORK / f"{kind}.msgpack" for kind in KINDS]
ext_paths = [WORK / f"ext_{kind}.msgpack" for kind in KINDS]
mixed_path = WORK / "mixed.msgpack"

pd.Series(
    {
        "columnar, all kinds": best_ms(lambda: [load_bopp_msgpack(p) for p in columnar_paths]),
        "ext per type, first pass only": best_ms(lambda: [load_bopp_msgpack(p, resolve_ext=False) for p in ext_paths]),
        "ext per type, both passes": best_ms(lambda: [load_bopp_msgpack(p) for p in ext_paths]),
        "mixed, first pass only": best_ms(lambda: load_bopp_msgpack(mixed_path, resolve_ext=False)),
        "mixed, first pass + decode_mixed": best_ms(lambda: decode_mixed(load_bopp_msgpack(mixed_path, resolve_ext=False))),
    },
    name="msgpack decode (ms)",
).round(3)

columnar, all kinds                 4.048
ext per type, first pass only       2.647
ext per type, both passes           2.704
mixed, first pass only              1.943
mixed, first pass + decode_mixed    2.323
Name: msgpack decode (ms), dtype: float64

On this run, summed over the five per-kind files: columnar is 57,809 B (JSON) / 21,434 B (MsgPack); ext per type is 78,515 B / 53,728 B; the single-file ext mixed form is 89,098 B / 60,459 B. Mixed is larger than the ext-per-type sum because every one of its 716 rows repeats an `event` tag that the per-type rows don't carry.

MsgPack decode time (best of 5 repeats of 20 calls): columnar 4.048 ms, ext per type 2.647 ms first pass only and 2.704 ms both passes — row validation added no measurable cost, the two are within run-to-run noise. The mixed form decoded fastest, 1.943 ms first pass alone and 2.323 ms with `decode_mixed`, despite holding all 716 events; one larger file likely beats five separate opens and decodes (unconfirmed). Unlike the notes part, where columnar decoded faster than ext, here columnar is slowest of the three — five files with many sparse nullable columns — an observation, not a profiled cause.

Registry finding: an entry point must resolve to a `type`, so looking one up for `ScoreControlEvent` (a `UnionType`) fails with "must be a type, got UnionType"; assigning the union directly (`REGISTRY[name] = ScoreControlEvent`) skips that check, and msgspec resolves the tagged union into `ChordEvent`, `TempoEvent`, and the rest. The restriction is the registry's type check on entry points, not msgspec: a heterogeneous ext payload can only be registered through an entry point as one schema per event type, unless the registry's type check is relaxed to accept unions.

Ids stay independent of the plugin: they hash the deterministic JSON encoding, identical whether rows are resolved structs or raw dicts, with or without the plugin installed. One wire-level caveat: resolved row structs with more than 15 fields (`ScoreChord` has 20) can produce different MsgPack *file* bytes than equivalent dict rows, since msgspec sizes the map header from the struct's field count (`map16` instead of `fixmap`); ids are unaffected, being computed from the JSON encoding.